# Phase 7 runner (thin) — experiment C (B + anatomical preservation)

Runs **existing repository scripts** on Kaggle. Design, pre-registered rules and what is compared: `docs/PHASE_7_ANATOMICAL_PRESERVATION.md`.
One session does everything: smoke run of C -> pre-registered smoke criteria -> lambda2 from the pre-registered rule (applied automatically, never chosen
by hand) -> lean C training (600 steps, B's recipe, lambda1 = 0.5 fixed) -> the same 24-image evaluation used in Phase 6 -> scoring.
A0, A-ft and B are NOT rerun: their Phase 6 scores are compared offline against C's.

Attach: the repo dataset, CheXpert-small (`ashery/chexpert`), and the dataset containing the Phase 5b `best.pt` (SHA-256 `d14a9e87...763a`; any other file is refused).
GPU T4 (one GPU is enough), Internet On. Edit only the parameter cell.

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, time, zipfile

# ---------------- parameters (edit only this cell) ----------------
RUN_SMOKE = True        # 20-step C smoke run (+ resume check) and the pre-registered smoke criteria; also yields lambda2 (rule R-C2)
RUN_TRAIN = True        # lean C training; starts only if the smoke criteria passed (or LAMBDA2 is given explicitly)
RUN_EVAL = True         # the same 24 held-out images as Phase 6, then scoring with the frozen classifier + classifier-free metrics
LAMBDA2 = None          # None = use the value computed by the pre-registered rule from this session's smoke run
MAX_STEPS = 600         # identical to Phase 6's A-ft and B
MICRO_BATCH, GRAD_ACCUM = 2, 4   # identical to Phase 6
PINNED_CLASSIFIER_SHA256 = "d14a9e87519e80803449cdeb2cf550a08b4508f52587d3767aa967f8cb51763a"
PRISM_CKPT = "/kaggle/temp/prism/model.safetensors"
RESTORE_FROM = sorted(p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + "experiments"))   # previous output, if attached
# ------------------------------------------------------------------

WORK, REPO = "/kaggle/working", "/kaggle/working/repo"
OUT = f"{WORK}/experiments"
P5, P6 = f"{OUT}/phase5", f"{OUT}/phase6"
os.makedirs(P5, exist_ok=True); os.makedirs(P6, exist_ok=True)
os.environ.update(HF_HOME="/kaggle/temp/hf", PYTHONUNBUFFERED="1")
FAILED, LOG_PATH = [], f"{OUT}/notebook_log.txt"
LOG = open(LOG_PATH, "a")

def say(text=""):
    print(text, flush=True); LOG.write(text + "\n"); LOG.flush()

def run(name, cmd):
    say(); say(f"=== {name}"); say("$ " + " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=REPO if os.path.isdir(REPO) else None, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        say(line.rstrip("\n"))
    rc = proc.wait(); say(f"--- exit code {rc}")
    if rc: FAILED.append((name, rc))
    return rc

In [ ]:
def find(suffix):
    return sorted((p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + suffix)), key=len)

say(f"notebook flags: SMOKE={RUN_SMOKE} TRAIN={RUN_TRAIN} EVAL={RUN_EVAL} LAMBDA2={LAMBDA2} MAX_STEPS={MAX_STEPS} micro_batch={MICRO_BATCH} accum={GRAD_ACCUM}")
boots = find("scripts/kaggle_bootstrap.py")
if not boots:
    zips = find("*repo*.zip")
    if zips:
        shutil.unpack_archive(zips[0], "/kaggle/temp/repo_src"); boots = ["/kaggle/temp/repo_src/scripts/kaggle_bootstrap.py"]
assert boots, "repo package not found under /kaggle/input: attach the dataset built by scripts/package_for_kaggle.py"
say(f"using repo package: {boots[0]}")
run("bootstrap (copy repo, install deps, torch guarded)", [sys.executable, boots[0], "--repo-src", os.path.dirname(os.path.dirname(boots[0])), "--workdir", REPO])
run("environment check", [sys.executable, "scripts/check_environment.py", "--require-gpu", "--min-vram-gb", "8", "--out", f"{OUT}/env_report.json"])
clf = [p for p in find("best.pt") if hashlib.sha256(open(p, "rb").read()).hexdigest() == PINNED_CLASSIFIER_SHA256]
assert clf, "no best.pt with SHA-256 " + PINNED_CLASSIFIER_SHA256 + " under /kaggle/input: attach the Phase 5b classifier checkpoint"
CLF = clf[0]; say(f"frozen classifier: {CLF} (hash verified)")
if not os.path.exists(PRISM_CKPT):
    run("fetch PRISM checkpoint (3.4 GB, size + SHA-256 verified)", [sys.executable, "scripts/download_prism_checkpoint.py", "--dest", os.path.dirname(PRISM_CKPT)])
LOCATED = run("locate CheXpert + write Phase 5 configs", [sys.executable, "scripts/locate_chexpert.py", "--out-dir", P5, "--prism-checkpoint", PRISM_CKPT]) == 0
if LOCATED:
    run("generated configs + the fixed 24-image evaluation set (same seed as Phase 6)", [sys.executable, "scripts/make_phase6_inputs.py", "--phase5-dir", P5, "--prism-checkpoint", PRISM_CKPT, "--classifier-checkpoint", CLF, "--out-dir", P6])
READY = os.path.exists(f"{P6}/generated_phase7_C.yaml")
EVAL_READY = os.path.exists(f"{P6}/eval_set.json")
SETS = ["--set", f"finetune.micro_batch={MICRO_BATCH}", "--set", f"finetune.grad_accum={GRAD_ACCUM}"]
def c_cmd(name, extra):
    return [sys.executable, "scripts/run_experiment.py", "--config", f"{P6}/generated_phase7_C.yaml", "--name", name, "--run-root", OUT, *SETS, *extra]

In [ ]:
SMOKE_OK = False
if RUN_SMOKE and READY:
    sm = ["--set", "finetune.max_steps=20", "--set", "finetune.save_every=10", "--set", "finetune.log_every=2", "--set", "losses.anatomy_preservation.weight=1.0"]
    run("smoke C (lambda2 = 1.0 is a smoke-only placeholder)", c_cmd("phase7_smoke_C", sm))
    run("smoke C resume (20 -> 24 steps)", c_cmd("phase7_smoke_C", ["--set", "finetune.max_steps=24", "--set", "finetune.save_every=10", "--set", "finetune.log_every=2",
                                                                 "--set", "losses.anatomy_preservation.weight=1.0", "--restore-from", OUT]))
    SMOKE_OK = run("smoke PASS/FAIL against the pre-registered criteria + rule R-C2 (lambda2)", [sys.executable, "scripts/check_phase7_smoke.py", "--run-root", OUT, "--out", f"{P6}/smoke_check_phase7.json"]) == 0
    if SMOKE_OK and LAMBDA2 is None:
        LAMBDA2 = json.load(open(f"{P6}/smoke_check_phase7.json"))["rules"]["R_C2_lambda2"]
    say(f"lambda2 for the lean run: {LAMBDA2} ({'from rule R-C2' if SMOKE_OK else 'smoke did not pass'})")

In [ ]:
TRAINED = False
if RUN_TRAIN and READY:
    if LAMBDA2 is None or (RUN_SMOKE and not SMOKE_OK):
        say("lean C training NOT started: the smoke criteria did not pass (or no lambda2 is available). See the smoke check above.")
    else:
        tr = ["--set", f"finetune.max_steps={MAX_STEPS}", "--set", f"losses.anatomy_preservation.weight={LAMBDA2}"] + (["--restore-from", *RESTORE_FROM] if RESTORE_FROM else [])
        TRAINED = run(f"train C (600 steps, lambda1 = 0.5 fixed, lambda2 = {LAMBDA2})", c_cmd("phase7_C", tr)) == 0

In [ ]:
def delta_of(name):
    c = sorted((p for root in [OUT, *RESTORE_FROM] for p in glob.glob(f"{root}/{name}/*/checkpoints/delta_final.safetensors")), key=lambda p: p.split("/")[-3])
    assert c, f"no trained delta for {name}: run RUN_TRAIN first"
    return c[-1]

if RUN_EVAL and EVAL_READY and (TRAINED or not RUN_TRAIN):
    cmd = [sys.executable, "scripts/run_experiment.py", "--config", f"{P6}/generated_eval.yaml", "--name", "phase7_eval_C", "--run-root", OUT,
           "--set", f"editing.inversion_cache_dir={OUT}/phase6/inversion_cache/C", "--set", f"model.delta_checkpoint_path={delta_of('phase7_C')}"]
    if run("eval C: 24 fixed test images, add Cardiomegaly, self_replace_steps 0.2", cmd) == 0:
        d = sorted(glob.glob(f"{OUT}/phase7_eval_C/*"))[-1]
        run("score C", [sys.executable, "scripts/evaluate_counterfactuals.py", "--run-dir", d, "--eval-set", f"{P6}/eval_set.json", "--classifier", CLF,
                        "--classifier-sha", PINNED_CLASSIFIER_SHA256, "--arm", "C", "--out-dir", f"{P6}/scores"])

In [ ]:
# Always archive (even after a failure). Checkpoints > 50 MB stay in /kaggle/working (SHA-256 in each run's artifacts.json) so the zip stays downloadable.
stamp = time.strftime('%Y%m%d-%H%M%S'); archive = f"{WORK}/artifacts_{stamp}.zip"; skipped = []
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            p = os.path.join(root, f)
            if f.endswith((".pt", ".safetensors")) and os.path.getsize(p) > 50e6:
                skipped.append(os.path.relpath(p, OUT)); continue
            if os.sep + "inversion_cache" + os.sep in p or f == "train_resolved.csv":
                continue
            z.write(p, os.path.relpath(p, OUT))
say(f"artifacts: {archive} {os.path.getsize(archive) / 1e6:.2f} MB; large checkpoints left out of the zip: {skipped}")
say(f"artifacts sha256: {hashlib.file_digest(open(archive, 'rb'), 'sha256').hexdigest()}   (verify the downloaded zip against this line)")
say(f"failed steps: {FAILED or 'none'}")
if FAILED:
    raise RuntimeError(f"failed steps: {FAILED}")